# Figure3a external activity


In [ ]:
import gc
import os
import time
import numpy as np
import pandas as pd
import anndata as ad
from tqdm import tqdm
from joblib import dump, load
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, confusion_matrix, f1_score, balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')

TRAIN_DIR = 'results/classification_activity/results_elasticnet_C1_l1_0.5'

ITN_H5AD = 'data/adata_cohort_2.h5ad'
NYU_H5AD = 'data/adata_cohort3.h5ad'

OUTPUT_DIR = 'results/classification_activity/results_elasticnet_C1_l1_0.5_external'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SLEDAI_HIGH = 11
SLEDAI_LOW  = 0

print("Loading trained models...")
models_dict = load(f'{TRAIN_DIR}/models.joblib')
print(f"  Loaded {len(models_dict)} models")

with open(f'{TRAIN_DIR}/feature_names.txt', 'r') as f:
    train_features = [line.strip() for line in f.readlines()]
print(f"  Training feature set: {len(train_features)} peptides")

config = load(f'{TRAIN_DIR}/config.joblib')
print(f"  Model: {config['model_type']}, C={config['C']}, "
      f"l1_ratio={config.get('l1_ratio', 'N/A')}")

def batch_predict_proba(models_dict, X_np):
    model_names = list(models_dict.keys())
    n_models = len(model_names)
    n_features = X_np.shape[1]

    coefs = np.zeros((n_models, n_features))
    intercepts = np.zeros(n_models)

    print(f"  Extracting coefficients from {n_models} models...")
    for i, (name, model) in enumerate(models_dict.items()):
        coefs[i] = model.coef_[0]
        intercepts[i] = model.intercept_[0]

    print(f"  Computing predictions (single matmul: {X_np.shape} @ {coefs.T.shape})...")
    t0 = time.time()
    logits = X_np @ coefs.T + intercepts
    all_probas = 1.0 / (1.0 + np.exp(-logits))
    print(f"  Done in {time.time() - t0:.2f}s")

    return all_probas.T, model_names

def evaluate_models_vectorized(train_results_dir, X_test, y_test, train_features,
                                cohort_name='External', donor_col_nunique=None,
                                verbose=True):
    t_start = time.time()

    models = load(os.path.join(train_results_dir, 'models.joblib'))

    train_results_df = pd.read_csv(os.path.join(train_results_dir, 'results.csv'))
    if 'optimal_threshold' not in train_results_df.columns:
        print("WARNING: 'optimal_threshold' not found — using 0.5")
        train_results_df['optimal_threshold'] = 0.5

    assert list(X_test.columns) == train_features, "Feature mismatch!"
    X_np = X_test.values
    y_np = np.array(y_test)

    all_probas, model_names = batch_predict_proba(models, X_np)

    results, roc_data = [], []
    all_thresholds = []
    predictions_data = {"y_true": y_np}

    if verbose:
        print(f"\n  Computing metrics for {len(model_names)} models...")

    for i, model_name in enumerate(tqdm(model_names, desc="  Metrics", disable=not verbose)):
        pred_proba = all_probas[i]

        fpr, tpr, thresholds = roc_curve(y_np, pred_proba)

        roc_data.append({
            'model': model_name, 'fpr': fpr, 'tpr': tpr,
            'thresholds': thresholds, 'y_true': y_np,
            'y_pred_proba': pred_proba
        })

        seed = int(model_name.split('_')[1])
        fold = int(model_name.split('_')[3])

        matching = train_results_df[
            (train_results_df['seed'] == seed) &
            (train_results_df['fold'] == fold)]
        optimal_threshold = matching['optimal_threshold'].values[0] if len(matching) > 0 else 0.5
        all_thresholds.append(optimal_threshold)

        pred_binary = (pred_proba >= optimal_threshold).astype(int)
        predictions_data[f"{model_name}_proba"] = pred_proba
        predictions_data[f"{model_name}_label"] = pred_binary

        tn, fp, fn, tp = confusion_matrix(y_np, pred_binary).ravel()
        auroc = roc_auc_score(y_np, pred_proba)
        auprc = average_precision_score(y_np, pred_proba)

        results.append({
            'model': model_name, 'seed': seed, 'fold': fold,
            'auroc': auroc, 'auprc': auprc,
            'optimal_threshold': optimal_threshold,
            'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
            'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
            'ppv': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
            'f1_score': f1_score(y_np, pred_binary),
            'balanced_accuracy': balanced_accuracy_score(y_np, pred_binary),
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
        })

    results_df = pd.DataFrame(results)

    ensemble_pred_proba = np.mean(all_probas, axis=0)
    ensemble_threshold = np.mean(all_thresholds)
    ensemble_threshold_std = np.std(all_thresholds)
    ensemble_pred_binary = (ensemble_pred_proba >= ensemble_threshold).astype(int)

    predictions_data["ensemble_proba"] = ensemble_pred_proba
    predictions_data["ensemble_label"] = ensemble_pred_binary
    predictions_df = pd.DataFrame(predictions_data, index=X_test.index)

    ensemble_cm = confusion_matrix(y_np, ensemble_pred_binary)
    tn, fp, fn, tp = ensemble_cm.ravel()
    ensemble_fpr, ensemble_tpr, ensemble_thresholds_roc = roc_curve(y_np, ensemble_pred_proba)

    ensemble_metrics = {
        'auroc': roc_auc_score(y_np, ensemble_pred_proba),
        'auprc': average_precision_score(y_np, ensemble_pred_proba),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'ppv': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'f1_score': f1_score(y_np, ensemble_pred_binary),
        'balanced_accuracy': balanced_accuracy_score(y_np, ensemble_pred_binary),
        'confusion_matrix': ensemble_cm,
        'optimal_threshold': ensemble_threshold,
        'optimal_threshold_std': ensemble_threshold_std,
        'fpr': ensemble_fpr, 'tpr': ensemble_tpr,
        'thresholds': ensemble_thresholds_roc
    }

    elapsed = time.time() - t_start

    if verbose:
        print(f"\n  Total evaluation time: {elapsed:.1f}s")
        print("\n" + "="*60)
        print(f"{cohort_name} — INDIVIDUAL MODEL PERFORMANCE SUMMARY:")
        print("="*60)

        print("\nThreshold-Independent Metrics (TRUSTWORTHY):")
        for metric in ['auroc', 'auprc']:
            print(f"  {metric.upper()}: {results_df[metric].mean():.3f} ± {results_df[metric].std():.3f}")

        print(f"\nThreshold-Dependent Metrics (training thresholds, "
              f"mean={ensemble_threshold:.3f} ± {ensemble_threshold_std:.3f}):")
        for metric in ['sensitivity', 'specificity', 'ppv', 'npv', 'f1_score', 'balanced_accuracy']:
            print(f"  {metric.upper()}: {results_df[metric].mean():.3f} ± {results_df[metric].std():.3f}")

        print("\n" + "="*60)
        print(f"{cohort_name} — ENSEMBLE MODEL PERFORMANCE:")
        print("="*60)
        print(f"  Ensemble Threshold: {ensemble_threshold:.3f} ± {ensemble_threshold_std:.3f}")
        print(f"  AUROC: {ensemble_metrics['auroc']:.3f}")
        print(f"  AUPRC: {ensemble_metrics['auprc']:.3f}")
        print(f"\nEnsemble Confusion Matrix:")
        print(ensemble_metrics['confusion_matrix'])
        print(f"\nTotal test samples: {len(X_test)}")
        if donor_col_nunique is not None:
            print(f"Total test donors: {donor_col_nunique}")

    return {
        'results_df': results_df,
        'ensemble_metrics': ensemble_metrics,
        'predictions_df': predictions_df,
        'roc_data': roc_data
    }

print("\n" + "#"*80)
print("ITN COHORT — HIGH vs LOW/NO DISEASE ACTIVITY")
print("#"*80)

adata_itn = ad.read_h5ad(ITN_H5AD)
adata_itn.obs = adata_itn.obs.rename(columns={'sample_ID': 'patient_ID'})

print(f"Loaded ITN: {adata_itn.shape}")
print(f"group distribution:")
print(adata_itn.obs['group'].value_counts(dropna=False))

adata_itn = adata_itn[adata_itn.obs['group'] == 'lupus'].copy()
print(f"\nAfter filtering to lupus only: {adata_itn.shape}")

print("\nhigh_vs_no distribution (before filtering NaN):")
print(adata_itn.obs['high_vs_no'].value_counts(dropna=False))

adata_itn = adata_itn[adata_itn.obs['high_vs_no'] != "nan"].copy()

print("\nAFTER FILTERING:")
print(adata_itn.obs['high_vs_no'].value_counts(dropna=False))
print(f"Total samples: {adata_itn.shape[0]}")
print(f"Unique donors (patient_ID): {adata_itn.obs['patient_ID'].nunique()}")

visit_counts = adata_itn.obs['patient_ID'].value_counts()
print(f"Visits per donor: {dict(visit_counts.value_counts().sort_index())}")

print("\nPreparing ITN features...")
t0 = time.time()

X_itn = adata_itn.to_df(layer="log_fold_change_over_AG")
X_itn[np.isnan(X_itn) | np.isinf(X_itn)] = 0

missing_itn = set(train_features) - set(X_itn.columns)
extra_itn = set(X_itn.columns) - set(train_features)
print(f"Feature alignment: {len(missing_itn)} missing (zero-filled), {len(extra_itn)} extra (dropped)")

if len(missing_itn) == 0 and len(extra_itn) == 0:
    X_itn_aligned = X_itn[train_features]
else:
    X_itn_aligned = pd.DataFrame(0.0, index=X_itn.index, columns=train_features)
    shared = list(set(train_features) & set(X_itn.columns))
    X_itn_aligned[shared] = X_itn[shared]

y_itn = (adata_itn.obs['high_vs_no'] == 'high_disease_activity').astype(int).values
print(f"Labels: {sum(y_itn)} high / {len(y_itn) - sum(y_itn)} low-or-no")
print(f"Feature prep done in {time.time() - t0:.1f}s")

itn_results = evaluate_models_vectorized(
    TRAIN_DIR, X_itn_aligned, y_itn, train_features,
    cohort_name='ITN',
    donor_col_nunique=adata_itn.obs['patient_ID'].nunique()
)

itn_results['results_df'].to_csv(f'{OUTPUT_DIR}/itn_test_results.csv', index=False)
itn_results['predictions_df'].to_csv(f'{OUTPUT_DIR}/itn_test_predictions.csv')
dump(itn_results['roc_data'], f'{OUTPUT_DIR}/itn_test_roc_data.joblib')
dump(itn_results['ensemble_metrics'], f'{OUTPUT_DIR}/itn_test_ensemble_metrics.joblib')
print(f"ITN results saved to {OUTPUT_DIR}/")

del adata_itn, X_itn, X_itn_aligned
gc.collect()

print("\n" + "#"*80)
print("NYU COHORT — HIGH vs LOW SLEDAI")
print("#"*80)

adata_nyu = ad.read_h5ad(NYU_H5AD)
print(f"Loaded NYU adata: {adata_nyu.shape}")
print(f"Unique donors (unique_subject_id): {adata_nyu.obs['unique_subject_id'].nunique()}")
print(adata_nyu.obs['group'].value_counts(dropna=False))

adata_nyu_sle = adata_nyu[adata_nyu.obs['group'] == 'lupus'].copy()
print(f"\nAfter filtering to SLE (lupus) only: {adata_nyu_sle.shape[0]}")

nyu_sledai = adata_nyu_sle.obs['sledai_score']
print(f"SLEDAI available: {nyu_sledai.notna().sum()} / {len(nyu_sledai)}")

nyu_mask = (nyu_sledai >= SLEDAI_HIGH) | (nyu_sledai == SLEDAI_LOW)
adata_nyu_filtered = adata_nyu_sle[nyu_mask].copy()

n_high = sum(adata_nyu_filtered.obs['sledai_score'] >= SLEDAI_HIGH)
n_low = sum(adata_nyu_filtered.obs['sledai_score'] == SLEDAI_LOW)
print(f"\nAfter SLEDAI filter: {adata_nyu_filtered.shape[0]} samples ({n_high} high, {n_low} low)")
print(f"Unique donors: {adata_nyu_filtered.obs['unique_subject_id'].nunique()}")

print("\nPreparing NYU features...")
t0 = time.time()

X_nyu = adata_nyu_filtered.to_df(layer="log_fold_change_over_AG")
X_nyu[np.isnan(X_nyu) | np.isinf(X_nyu)] = 0

missing_nyu = set(train_features) - set(X_nyu.columns)
extra_nyu = set(X_nyu.columns) - set(train_features)
print(f"Feature alignment: {len(missing_nyu)} missing (zero-filled), {len(extra_nyu)} extra (dropped)")

if len(missing_nyu) == 0 and len(extra_nyu) == 0:
    X_nyu_aligned = X_nyu[train_features]
else:
    X_nyu_aligned = pd.DataFrame(0.0, index=X_nyu.index, columns=train_features)
    shared = list(set(train_features) & set(X_nyu.columns))
    X_nyu_aligned[shared] = X_nyu[shared]

y_nyu = (adata_nyu_filtered.obs['sledai_score'] >= SLEDAI_HIGH).astype(int).values
print(f"Labels: {sum(y_nyu)} high / {len(y_nyu) - sum(y_nyu)} low")
print(f"Feature prep done in {time.time() - t0:.1f}s")

nyu_results = evaluate_models_vectorized(
    TRAIN_DIR, X_nyu_aligned, y_nyu, train_features,
    cohort_name='NYU',
    donor_col_nunique=adata_nyu_filtered.obs['unique_subject_id'].nunique()
)

nyu_results['results_df'].to_csv(f'{OUTPUT_DIR}/nyu_test_results.csv', index=False)
nyu_results['predictions_df'].to_csv(f'{OUTPUT_DIR}/nyu_test_predictions.csv')
dump(nyu_results['roc_data'], f'{OUTPUT_DIR}/nyu_test_roc_data.joblib')
dump(nyu_results['ensemble_metrics'], f'{OUTPUT_DIR}/nyu_test_ensemble_metrics.joblib')
print(f"NYU results saved to {OUTPUT_DIR}/")

del adata_nyu, adata_nyu_sle, adata_nyu_filtered, X_nyu, X_nyu_aligned
gc.collect()

def plot_roc_validation_combined_highlow(
    nyu_results_df, nyu_roc_data,
    itn_results_df, itn_roc_data,
    save_plot=False,
    output_path=None,
    figsize=(7, 7),
    ci_level=0.95,
    decimal_places=2,
):

    nyu_color = "#90719f"
    itn_color = "#466c4b"

    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['axes.linewidth'] = 1.2
    plt.rcParams['xtick.major.width'] = 1.2
    plt.rcParams['ytick.major.width'] = 1.2

    mean_fpr = np.linspace(0, 1, 1000)
    alpha_low = (1 - ci_level) / 2 * 100
    alpha_high = (1 + ci_level) / 2 * 100
    fmt = f'.{decimal_places}f'

    fig, ax = plt.subplots(figsize=figsize)
    stats_dict = {}

    tprs_nyu = []
    for data in nyu_roc_data:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs_nyu.append(interp_tpr)
    tprs_nyu = np.array(tprs_nyu)
    mean_tpr_nyu = np.mean(tprs_nyu, axis=0)
    mean_tpr_nyu[-1] = 1.0

    aucs_nyu = nyu_results_df['auroc'].values
    mean_auc_nyu = np.mean(aucs_nyu)
    ci_low_nyu = np.percentile(aucs_nyu, alpha_low)
    ci_high_nyu = np.percentile(aucs_nyu, alpha_high)

    tprs_lower_nyu = np.percentile(tprs_nyu, alpha_low, axis=0)
    tprs_upper_nyu = np.percentile(tprs_nyu, alpha_high, axis=0)

    nyu_label = (f'NYU cohort: AUC = {mean_auc_nyu:{fmt}} '
                 f'({ci_low_nyu:{fmt}}\u2013{ci_high_nyu:{fmt}})')

    ax.fill_between(mean_fpr, tprs_lower_nyu, tprs_upper_nyu,
                    color=nyu_color, alpha=0.15, linewidth=0, zorder=2)
    ax.plot(mean_fpr, mean_tpr_nyu, color=nyu_color, linewidth=2.5,
            label=nyu_label, zorder=4)

    stats_dict['nyu'] = {
        'mean_auc': mean_auc_nyu,
        'ci_low_auc': ci_low_nyu,
        'ci_high_auc': ci_high_nyu,
        'n_models': len(aucs_nyu),
    }

    tprs_itn = []
    for data in itn_roc_data:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs_itn.append(interp_tpr)
    tprs_itn = np.array(tprs_itn)
    mean_tpr_itn = np.mean(tprs_itn, axis=0)
    mean_tpr_itn[-1] = 1.0

    aucs_itn = itn_results_df['auroc'].values
    mean_auc_itn = np.mean(aucs_itn)
    ci_low_itn = np.percentile(aucs_itn, alpha_low)
    ci_high_itn = np.percentile(aucs_itn, alpha_high)

    tprs_lower_itn = np.percentile(tprs_itn, alpha_low, axis=0)
    tprs_upper_itn = np.percentile(tprs_itn, alpha_high, axis=0)

    itn_label = (f'ITN cohort: AUC = {mean_auc_itn:{fmt}} '
                 f'({ci_low_itn:{fmt}}\u2013{ci_high_itn:{fmt}})')

    ax.fill_between(mean_fpr, tprs_lower_itn, tprs_upper_itn,
                    color=itn_color, alpha=0.15, linewidth=0, zorder=2)
    ax.plot(mean_fpr, mean_tpr_itn, color=itn_color, linewidth=2.5,
            label=itn_label, zorder=4)

    stats_dict['itn'] = {
        'mean_auc': mean_auc_itn,
        'ci_low_auc': ci_low_itn,
        'ci_high_auc': ci_high_itn,
        'n_models': len(aucs_itn),
    }

    ax.plot([0, 1], [0, 1], color='gray', linewidth=1.5, linestyle='--',
            alpha=0.7, label='Random classifier', zorder=1)

    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('1 \u2212 Specificity', fontsize=15, fontweight='medium')
    ax.set_ylabel('Sensitivity', fontsize=15, fontweight='medium')
    ax.tick_params(axis='both', which='major', labelsize=12, length=5)
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, zorder=0)
    ax.set_aspect('equal', adjustable='box')

    legend = ax.legend(loc='lower right', fontsize=13, framealpha=0.95,
                       edgecolor='gray', fancybox=False)
    legend.get_frame().set_linewidth(0.8)

    plt.tight_layout()

    if save_plot and output_path:
        plt.savefig(f'{output_path}.pdf', bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"\nFigure saved to: {output_path}.pdf")

    print("\n" + "="*60)
    print("EXTERNAL VALIDATION ROC STATISTICS (High vs Low/No)")
    print("="*60)

    print(f"\nNYU Validation Cohort:")
    print(f"  Mean AUC: {mean_auc_nyu:.4f}")
    print(f"  95% CI:   [{ci_low_nyu:.4f}, {ci_high_nyu:.4f}]")
    print(f"  N models: {len(aucs_nyu)}")

    print(f"\nITN Validation Cohort:")
    print(f"  Mean AUC: {mean_auc_itn:.4f}")
    print(f"  95% CI:   [{ci_low_itn:.4f}, {ci_high_itn:.4f}]")
    print(f"  N models: {len(aucs_itn)}")

    print("\n" + "="*60)

    plt.show()
    return fig, ax, stats_dict

fig, ax, stats = plot_roc_validation_combined_highlow(
    nyu_results_df=nyu_results['results_df'],
    nyu_roc_data=nyu_results['roc_data'],
    itn_results_df=itn_results['results_df'],
    itn_roc_data=itn_results['roc_data'],
    save_plot=True,
    output_path=f'{OUTPUT_DIR}/nyu_itn_highlow_test_roc',
    figsize=(7, 7),
    ci_level=0.95,
    decimal_places=2,
)